# **FX Carry Project**

## G10, MXN, BRL, HKD, CNY, and CNH historical performance

This notebook builds a clean first-pass FX research framework around three questions:

1. Which currencies have delivered the strongest and weakest spot performance versus USD?
2. When do correlations rise or break down across currencies?
3. How much of the historical pattern lines up with risk appetite, USD strength, rates, carry, and implied volatility?

The analysis is intentionally quant-research style but still explainable. We focus on transparent transformations: normalized spot returns, drawdowns, rolling correlations, regime correlations, macro/risk-factor correlations, and simple interest-rate carry proxies.


### Research takeaways to keep in mind

- FX quote conventions matter. Some Bloomberg tickers rise when the foreign currency strengthens, while others rise when USD strengthens. This notebook normalizes all spot series so that **higher always means the local currency strengthened versus USD**.
- MXN and BRL are usually higher-beta currencies. They can benefit from carry and commodity/risk-on conditions, but they tend to suffer during USD squeezes and volatility shocks.
- HKD should look unusually stable because it is managed inside a USD-linked band. Big moves in HKD are less about free-floating FX and more about liquidity, rates, and pressure inside the peg.
- CNY/CNH should show lower realized volatility than most free-floating currencies because China manages the exchange-rate regime. CNH may move more freely than onshore CNY during stress.
- JPY and CHF often behave differently from high-beta FX. They can act more defensively during risk-off periods, although this relationship changes when rate differentials dominate.


### 1. Setup


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:,.4f}".format)

def display_table(df, formatters=None, gradient=False):
    formatters = formatters or {}

    try:
        styled = df.style
        if gradient:
            styled = styled.background_gradient(cmap="RdBu", vmin=-1, vmax=1)
        if formatters:
            styled = styled.format(formatters)
        display(styled)
    except Exception:
        out = df.copy()
        for col, fmt in formatters.items():
            if col in out:
                out[col] = out[col].map(lambda x: fmt.format(x) if pd.notna(x) else "")
        display(out)

try:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"
except Exception:
    pass


### 2. Load the raw datasets


In [2]:
RAW = Path("/Users/arjunpatel/GitRepositories/FX_Carry_26_Summer_PL/data/raw")

TARGET_CCYS = [
    "EUR", "JPY", "GBP", "CHF", "CAD", "AUD", "NZD", "SEK", "NOK",
    "MXN", "BRL", "HKD", "CNY", "CNH"
]

FILES = {
    "g10_fx": "g10_fx_spot_forward_long.parquet",
    "em_fx": "em_fx_spot_forward_long.parquet",
    "g10_rates": "g10_interest_rates_long.parquet",
    "em_rates": "em_interest_rates_long.parquet",
    "em_onshore_rates": "em_onshore_rates_long.parquet",
    "g10_options": "g10_fx_options_long.parquet",
    "em_options": "em_fx_options_long.parquet",
    "global_risk": "global_risk_long.parquet",
    "em_risk": "em_risk_long.parquet",
    "macro_proxies": "macro_market_proxies_long.parquet",
    "usd_riskfree": "usd_riskfree_long.parquet",
}

def load_long_parquet(file_name):
    df = pd.read_parquet(RAW / file_name)
    df["date"] = pd.to_datetime(df["date"])
    df["ticker"] = df["ticker"].astype(str)
    df["field"] = df["field"].astype(str)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.sort_values(["date", "ticker", "field"]).reset_index(drop=True)

data = {name: load_long_parquet(file_name) for name, file_name in FILES.items()}

coverage = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(df),
            "tickers": df["ticker"].nunique(),
            "fields": ", ".join(sorted(df["field"].unique())),
            "start": df["date"].min(),
            "end": df["date"].max(),
        }
        for name, df in data.items()
    ]
).set_index("dataset")

display(coverage)


,rows,tickers,fields,start,end
dataset,,,,,
g10_fx,1007085,66,"PX_ASK, PX_BID, PX_LAST",2007-01-01,2026-06-30
em_fx,1393179,95,"PX_ASK, PX_BID, PX_LAST",2007-01-01,2026-06-30
g10_rates,365973,81,PX_LAST,2007-01-01,2026-06-30
em_rates,134365,34,PX_LAST,2007-01-01,2026-06-30
em_onshore_rates,87564,16,PX_LAST,2007-01-01,2026-07-06
g10_options,4170477,275,"PX_ASK, PX_BID, PX_LAST",2007-01-01,2026-06-30
em_options,4847571,325,"PX_ASK, PX_BID, PX_LAST",2007-01-01,2026-06-30
global_risk,95156,19,PX_LAST,2007-01-01,2026-06-30
em_risk,5091,1,PX_LAST,2007-01-01,2026-07-06


### 3. Define the currency universe and quote convention

The raw spot files include spot and forward-tenor tickers. For spot performance, we only want tickers such as `AUD Curncy`, `MXN Curncy`, and `CNH Curncy`, not `AUD3M Curncy` or `MXN12M Curncy`.

Bloomberg convention is mixed:

- `EUR`, `GBP`, `AUD`, and `NZD` are quoted as USD per foreign currency. If the raw spot price rises, the foreign currency strengthened.
- Most of the rest are quoted as local currency per USD. If the raw spot price rises, USD strengthened and the foreign currency weakened.

The function below flips the second group so every return series reads the same way: **positive return = local currency appreciated versus USD**.


In [3]:
CCY_LABELS = {
    "EUR": "Euro",
    "JPY": "Japanese yen",
    "GBP": "British pound",
    "CHF": "Swiss franc",
    "CAD": "Canadian dollar",
    "AUD": "Australian dollar",
    "NZD": "New Zealand dollar",
    "SEK": "Swedish krona",
    "NOK": "Norwegian krone",
    "MXN": "Mexican peso",
    "BRL": "Brazilian real",
    "HKD": "Hong Kong dollar",
    "CNY": "Chinese yuan, onshore",
    "CNH": "Chinese yuan, offshore",
}

DIRECT_USD_PER_FX = {"EUR", "GBP", "AUD", "NZD"}
INVERSE_FX_PER_USD = set(TARGET_CCYS) - DIRECT_USD_PER_FX

SPOT_TICKERS = {ccy: f"{ccy} Curncy" for ccy in TARGET_CCYS}
TICKER_TO_CCY = {ticker: ccy for ccy, ticker in SPOT_TICKERS.items()}

fx_raw = pd.concat([data["g10_fx"], data["em_fx"]], ignore_index=True)

spot_long = fx_raw[
    (fx_raw["field"] == "PX_LAST")
    & (fx_raw["ticker"].isin(SPOT_TICKERS.values()))
].copy()

spot_wide = (
    spot_long
    .pivot_table(index="date", columns="ticker", values="value", aggfunc="last")
    .rename(columns=TICKER_TO_CCY)
    .reindex(columns=TARGET_CCYS)
    .sort_index()
)

log_raw_spot = np.log(spot_wide)
spot_log_returns = log_raw_spot.diff()

for ccy in INVERSE_FX_PER_USD:
    if ccy in spot_log_returns:
        spot_log_returns[ccy] = -spot_log_returns[ccy]

spot_returns = np.expm1(spot_log_returns)
currency_index = (1 + spot_returns).cumprod() * 100
currency_index = currency_index.where(spot_returns.notna().cummax())

spot_coverage = pd.DataFrame({
    "label": pd.Series(CCY_LABELS),
    "raw_ticker": pd.Series(SPOT_TICKERS),
    "quote_type": [
        "USD per FX" if ccy in DIRECT_USD_PER_FX else "FX per USD, inverted"
        for ccy in TARGET_CCYS
    ],
    "first_date": spot_wide.apply(lambda s: s.first_valid_index()),
    "last_date": spot_wide.apply(lambda s: s.last_valid_index()),
    "observations": spot_wide.notna().sum(),
}).loc[TARGET_CCYS]

display(spot_coverage)


,label,raw_ticker,quote_type,first_date,last_date,observations
EUR,Euro,EUR Curncy,USD per FX,2007-01-01,2026-06-30,5087
JPY,Japanese yen,JPY Curncy,"FX per USD, inverted",2007-01-01,2026-06-30,5087
GBP,British pound,GBP Curncy,USD per FX,2007-01-01,2026-06-30,5087
CHF,Swiss franc,CHF Curncy,"FX per USD, inverted",2007-01-01,2026-06-30,5087
CAD,Canadian dollar,CAD Curncy,"FX per USD, inverted",2007-01-01,2026-06-30,5087
AUD,Australian dollar,AUD Curncy,USD per FX,2007-01-01,2026-06-30,5087
NZD,New Zealand dollar,NZD Curncy,USD per FX,2007-01-01,2026-06-30,5087
SEK,Swedish krona,SEK Curncy,"FX per USD, inverted",2007-01-01,2026-06-30,5087
NOK,Norwegian krone,NOK Curncy,"FX per USD, inverted",2007-01-01,2026-06-30,5086
MXN,Mexican peso,MXN Curncy,"FX per USD, inverted",2007-01-01,2026-06-30,5085


### 4. Spot performance versus USD


In [4]:
plot_df = (
    currency_index
    .resample("M").last()
    .reset_index()
    .melt(id_vars="date", var_name="currency", value_name="index")
    .dropna()
)

fig = px.line(
    plot_df,
    x="date",
    y="index",
    color="currency",
    hover_data={"index": ":.2f"},
    title="Normalized Spot Performance vs USD, 100 = Start of Each Series",
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    yaxis_title="Local currency strength vs USD",
    xaxis_title=None,
    legend_title_text="Currency",
)
fig


In [5]:
def max_drawdown(index_series):
    running_max = index_series.cummax()
    drawdown = index_series / running_max - 1
    return drawdown.min()

stats = []

for ccy in TARGET_CCYS:
    r = spot_log_returns[ccy].dropna()
    idx = currency_index[ccy].dropna()

    if len(r) < 252 or idx.empty:
        continue

    ann_return = np.exp(r.mean() * 252) - 1
    ann_vol = r.std() * np.sqrt(252)
    sharpe_like = ann_return / ann_vol if ann_vol != 0 else np.nan

    stats.append({
        "currency": ccy,
        "name": CCY_LABELS[ccy],
        "ann_spot_return": ann_return,
        "ann_vol": ann_vol,
        "return_to_vol": sharpe_like,
        "max_drawdown": max_drawdown(idx),
        "best_day": np.expm1(r.max()),
        "worst_day": np.expm1(r.min()),
        "positive_days": (r > 0).mean(),
        "obs": len(r),
    })

spot_stats = (
    pd.DataFrame(stats)
    .set_index("currency")
    .sort_values("return_to_vol", ascending=False)
)

display_table(
    spot_stats,
    {
        "ann_spot_return": "{:.2%}",
        "ann_vol": "{:.2%}",
        "return_to_vol": "{:.2f}",
        "max_drawdown": "{:.2%}",
        "best_day": "{:.2%}",
        "worst_day": "{:.2%}",
        "positive_days": "{:.1%}",
        "obs": "{:,.0f}",
    },
)


,name,ann_spot_return,ann_vol,return_to_vol,max_drawdown,best_day,worst_day,positive_days,obs
currency,,,,,,,,,
CNY,"Chinese yuan, onshore",0.91%,3.11%,0.29,-14.79%,1.62%,-1.82%,49.4%,"4,708"
CHF,Swiss franc,2.06%,10.34%,0.20,-30.02%,21.39%,-8.69%,48.3%,"5,086"
CNH,"Chinese yuan, offshore",-0.10%,4.16%,-0.02,-19.27%,2.02%,-2.71%,50.2%,"4,128"
AUD,Australian dollar,-0.65%,12.41%,-0.05,-47.89%,8.63%,-7.03%,51.2%,"5,086"
HKD,Hong Kong dollar,-0.03%,0.62%,-0.05,-1.28%,0.40%,-0.27%,45.9%,"5,082"
EUR,Euro,-0.71%,8.76%,-0.08,-40.00%,3.51%,-2.40%,49.8%,"5,086"
NZD,New Zealand dollar,-1.06%,12.24%,-0.09,-39.80%,4.42%,-6.52%,50.3%,"5,086"
CAD,Canadian dollar,-0.97%,8.65%,-0.11,-36.87%,4.08%,-3.20%,48.8%,"5,086"
SEK,Swedish krona,-1.71%,11.74%,-0.15,-48.61%,5.11%,-4.13%,50.2%,"5,086"


In [6]:
scatter_df = spot_stats.reset_index()
scatter_df["max_drawdown_abs"] = scatter_df["max_drawdown"].abs()
scatter_df["group"] = np.where(scatter_df["currency"].isin(["MXN", "BRL", "CNY", "CNH", "HKD"]), "Focus EM/Asia", "G10")

fig = px.scatter(
    scatter_df,
    x="ann_vol",
    y="ann_spot_return",
    color="group",
    size="max_drawdown_abs",
    text="currency",
    hover_data={
        "name": True,
        "return_to_vol": ":.2f",
        "max_drawdown": ":.2%",
        "ann_vol": ":.2%",
        "ann_spot_return": ":.2%",
        "max_drawdown_abs": False,
    },
    title="Spot Return vs Volatility",
)
fig.update_traces(textposition="top center")
fig.update_layout(
    template="plotly_white",
    xaxis_title="Annualized volatility",
    yaxis_title="Annualized spot return",
    legend_title_text="Universe",
)
fig


**How to read this section**

This is spot-only performance, not total carry P&L. A currency can have weak spot performance but still be attractive in a carry strategy if the interest-rate pickup is large enough. That distinction is especially important for MXN and BRL. Conversely, low-yield defensive currencies can look strong in stress episodes because investors unwind risky positions and buy funding/safe-haven FX.


### 5. Calendar-year returns


In [7]:
yearly_returns = np.expm1(spot_log_returns.resample("Y").sum())
yearly_returns.index = yearly_returns.index.year

fig = px.imshow(
    yearly_returns.T * 100,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    labels={"x": "Year", "y": "Currency", "color": "Return (%)"},
    title="Calendar-Year Spot Returns vs USD",
    text_auto=".1f",
)
fig.update_layout(template="plotly_white")
fig


Calendar-year returns are useful because FX trends often come in macro regimes rather than smooth long-run drifts. Large clusters usually line up with changes in the Fed cycle, commodity prices, risk appetite, China growth expectations, or policy regime shifts.


### 6. Correlations by horizon


In [8]:
returns_by_horizon = {
    "Daily": spot_returns,
    "Monthly": np.expm1(spot_log_returns.resample("M").sum()),
    "Quarterly": np.expm1(spot_log_returns.resample("Q").sum()),
}

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=list(returns_by_horizon.keys()),
    horizontal_spacing=0.06,
)

for i, (horizon, ret_df) in enumerate(returns_by_horizon.items(), start=1):
    corr = ret_df.corr()
    fig.add_trace(
        go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.index,
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            reversescale=True,
            colorbar=dict(title="Corr") if i == 3 else None,
            showscale=(i == 3),
            hovertemplate="x=%{x}<br>y=%{y}<br>corr=%{z:.2f}<extra></extra>",
        ),
        row=1,
        col=i,
    )

fig.update_layout(
    title="Currency Return Correlations Across Sampling Horizons",
    template="plotly_white",
    height=520,
)
fig


In [9]:
def average_pairwise_corr(ret_df):
    corr = ret_df.dropna(how="all").corr()
    if corr.shape[0] < 2:
        return np.nan
    mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    return pd.Series(corr.where(mask).stack()).mean()

summary_rows = []

for horizon, ret_df in returns_by_horizon.items():
    summary_rows.append({
        "horizon": horizon,
        "average_pairwise_corr": average_pairwise_corr(ret_df),
        "observations": len(ret_df.dropna(how="all")),
    })

horizon_corr_summary = pd.DataFrame(summary_rows).set_index("horizon")

display_table(
    horizon_corr_summary,
    {
        "average_pairwise_corr": "{:.2f}",
        "observations": "{:,.0f}",
    },
)


,average_pairwise_corr,observations
horizon,,
Daily,0.36,"5,086"
Monthly,0.44,234
Quarterly,0.45,78


**Why horizons matter**

Daily correlations are noisy and often reflect liquidity shocks. Monthly and quarterly correlations are closer to macro behavior. If correlations rise with the horizon, the currencies are sharing a common macro trend. If daily correlations spike but longer-horizon correlations remain lower, the move is more likely a short-lived risk shock or positioning unwind.


### 7. Rolling and regime correlations


In [10]:
def rolling_average_pairwise_corr(ret_df, window=126, min_obs=80):
    values = []
    dates = []

    for end in range(window, len(ret_df) + 1):
        sample = ret_df.iloc[end - window:end]
        sample = sample.dropna(axis=1, thresh=min_obs)

        if sample.shape[1] < 2:
            continue

        dates.append(ret_df.index[end - 1])
        values.append(average_pairwise_corr(sample))

    return pd.Series(values, index=dates, name=f"{window}d avg pairwise corr")

rolling_corr_6m = rolling_average_pairwise_corr(spot_returns, window=126, min_obs=80)
rolling_corr_1y = rolling_average_pairwise_corr(spot_returns, window=252, min_obs=160)

rolling_corr_df = pd.concat([rolling_corr_6m, rolling_corr_1y], axis=1).reset_index(names="date")

fig = px.line(
    rolling_corr_df,
    x="date",
    y=rolling_corr_df.columns[1:],
    title="Rolling Average Pairwise Correlation",
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    yaxis_title="Average pairwise correlation",
    xaxis_title=None,
    legend_title_text="Window",
)
fig


In [11]:
REGIMES = {
    "GFC / USD funding stress": ("2007-07-01", "2009-06-30"),
    "Post-GFC / euro crisis": ("2009-07-01", "2012-12-31"),
    "Taper and commodity drawdown": ("2013-01-01", "2015-12-31"),
    "Pre-COVID expansion": ("2016-01-01", "2019-12-31"),
    "COVID shock and rebound": ("2020-01-01", "2021-12-31"),
    "Inflation and Fed hiking shock": ("2022-01-01", "2022-12-31"),
    "Post-hiking cycle": ("2023-01-01", "2026-06-30"),
}

regime_rows = []

for regime, (start, end) in REGIMES.items():
    ret = spot_returns.loc[start:end]
    corr = ret.corr()
    mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    pair_corrs = corr.where(mask).stack()

    regime_rows.append({
        "regime": regime,
        "start": start,
        "end": end,
        "obs": len(ret.dropna(how="all")),
        "average_corr": pair_corrs.mean(),
        "max_pair_corr": pair_corrs.max(),
        "min_pair_corr": pair_corrs.min(),
    })

regime_corr = pd.DataFrame(regime_rows).set_index("regime")

display_table(
    regime_corr,
    {
        "average_corr": "{:.2f}",
        "max_pair_corr": "{:.2f}",
        "min_pair_corr": "{:.2f}",
        "obs": "{:,.0f}",
    },
)

fig = px.bar(
    regime_corr.reset_index(),
    x="regime",
    y="average_corr",
    title="Average Currency Correlation by Macro Regime",
)
fig.update_layout(
    template="plotly_white",
    xaxis_title=None,
    yaxis_title="Average pairwise correlation",
)
fig.update_xaxes(tickangle=-30)
fig


,start,end,obs,average_corr,max_pair_corr,min_pair_corr
regime,,,,,,
GFC / USD funding stress,2007-07-01,2009-06-30,522,0.31,0.88,-0.41
Post-GFC / euro crisis,2009-07-01,2012-12-31,914,0.36,0.87,-0.16
Taper and commodity drawdown,2013-01-01,2015-12-31,783,0.26,0.74,-0.03
Pre-COVID expansion,2016-01-01,2019-12-31,"1,043",0.35,0.78,-0.02
COVID shock and rebound,2020-01-01,2021-12-31,523,0.41,0.89,-0.05
Inflation and Fed hiking shock,2022-01-01,2022-12-31,260,0.48,0.92,-0.00
Post-hiking cycle,2023-01-01,2026-06-30,912,0.45,0.88,0.04


**Interpretation**

Correlation usually rises in periods when the market is trading one big theme: USD funding stress, global recession risk, synchronized central-bank tightening, or a commodity shock. When the macro backdrop is calmer, country-specific rates, local policy, trade exposure, and idiosyncratic positioning matter more, so correlations tend to fall.


### 8. Macro and risk-factor correlations


In [12]:
global_risk = data["global_risk"]

risk_levels = (
    global_risk[global_risk["field"] == "PX_LAST"]
    .pivot_table(index="date", columns="ticker", values="value", aggfunc="last")
    .sort_index()
)

log_change_tickers = [
    "DXY Curncy",
    "SPX Index",
    "MXWO Index",
    "MXEF Index",
    "BCOM Index",
    "CL1 Comdty",
    "CO1 Comdty",
    "XAU Curncy",
    "HG1 Comdty",
]

level_change_tickers = [
    "VIX Index",
    "MOVE Index",
    "JPMVXYGL Index",
    "JPMVXYG7 Index",
    "JPMVXYEM Index",
    "USGG2YR Index",
    "USGG10YR Index",
    "USYC2Y10 Index",
]

risk_changes = pd.DataFrame(index=risk_levels.index)

for ticker in log_change_tickers:
    if ticker in risk_levels:
        risk_changes[ticker] = np.log(risk_levels[ticker]).diff()

for ticker in level_change_tickers:
    if ticker in risk_levels:
        risk_changes[ticker] = risk_levels[ticker].diff()

factor_labels = {
    "DXY Curncy": "DXY return",
    "SPX Index": "S&P 500 return",
    "MXWO Index": "World equities return",
    "MXEF Index": "EM equities return",
    "BCOM Index": "Commodities return",
    "CL1 Comdty": "WTI oil return",
    "CO1 Comdty": "Brent oil return",
    "XAU Curncy": "Gold return",
    "HG1 Comdty": "Copper return",
    "VIX Index": "VIX change",
    "MOVE Index": "MOVE change",
    "JPMVXYGL Index": "Global FX vol change",
    "JPMVXYG7 Index": "G7 FX vol change",
    "JPMVXYEM Index": "EM FX vol change",
    "USGG2YR Index": "US 2Y yield change",
    "USGG10YR Index": "US 10Y yield change",
    "USYC2Y10 Index": "US 2s10s change",
}

risk_changes = risk_changes.rename(columns=factor_labels)

joined = spot_log_returns.join(risk_changes, how="inner")
factor_cols = [col for col in risk_changes.columns if col in joined.columns]
factor_corr = joined.corr().loc[TARGET_CCYS, factor_cols]

fig = px.imshow(
    factor_corr,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    labels={"x": "Risk or macro factor", "y": "Currency", "color": "Corr"},
    title="Daily Correlation of Currency Returns with Risk and Macro Factors",
    text_auto=".2f",
)
fig.update_layout(template="plotly_white", height=650)
fig


/Users/arjunpatel/opt/anaconda3/envs/finm/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

invalid value encountered in log



In [13]:
key_factors = [c for c in ["DXY return", "VIX change", "EM equities return", "Commodities return", "US 2Y yield change"] if c in factor_corr.columns]

factor_corr_table = (
    factor_corr[key_factors]
    .sort_values(key_factors[0])
)

display_table(
    factor_corr_table,
    {col: "{:.2f}" for col in factor_corr_table.columns},
    gradient=True,
)


,DXY return,VIX change,EM equities return,Commodities return,US 2Y yield change
EUR,-0.92,-0.15,0.23,0.29,-0.14
SEK,-0.77,-0.28,0.33,0.35,-0.07
NOK,-0.71,-0.34,0.38,0.47,-0.05
GBP,-0.69,-0.21,0.29,0.31,-0.10
CHF,-0.68,0.04,0.09,0.19,-0.27
NZD,-0.58,-0.40,0.41,0.38,-0.04
AUD,-0.58,-0.46,0.48,0.45,0.01
CAD,-0.54,-0.43,0.40,0.48,0.05
JPY,-0.42,0.31,-0.18,-0.06,-0.49
CNH,-0.42,-0.18,0.27,0.22,-0.15


**What this should show**

- A higher DXY usually lines up with broad non-USD weakness, so most normalized currency returns should have negative correlation with DXY.
- VIX and FX-vol increases usually hurt high-beta currencies such as MXN, BRL, AUD, NZD, NOK, and sometimes CAD.
- Commodity strength should matter more for AUD, CAD, NOK, BRL, and MXN than for EUR, CHF, HKD, or CNY.
- JPY and CHF can stand apart in risk-off regimes because they are often treated as defensive or funding currencies, although rate differentials can overpower that behavior.


### 9. Interest-rate and carry proxy


In [14]:
all_rates = pd.concat(
    [
        data["g10_rates"],
        data["em_rates"],
        data["em_onshore_rates"],
        data["usd_riskfree"],
    ],
    ignore_index=True,
)

# Representative short-rate proxies. These are simple research proxies, not a full funded carry P&L.
RATE_TICKER_BY_CCY = {
    "USD": "US0003M Index",
    "EUR": "EUR003M Index",
    "GBP": "BP0003M Index",
    "JPY": "JY0003M Index",
    "CHF": "SF0003M Index",
    "AUD": "ADBB3M Curncy",
    "CAD": "CDOR03 Index",
    "NOK": "NIBOR3M Index",
    "SEK": "STIB3M Index",
    "HKD": "HDDRC Curncy",
    "MXN": "MXIBTIIE Index",
    "BRL": "BCCDIO Curncy",
    "CNY": "SHIF3M Index",
    "CNH": "CHSWPC Index",
}

available_rate_map = {
    ccy: ticker
    for ccy, ticker in RATE_TICKER_BY_CCY.items()
    if ticker in set(all_rates["ticker"].unique())
}

rate_long = all_rates[
    (all_rates["field"] == "PX_LAST")
    & (all_rates["ticker"].isin(available_rate_map.values()))
].copy()

rate_wide = (
    rate_long
    .pivot_table(index="date", columns="ticker", values="value", aggfunc="last")
    .rename(columns={ticker: ccy for ccy, ticker in available_rate_map.items()})
    .sort_index()
    .ffill()
)

display(pd.Series(available_rate_map, name="rate_ticker").to_frame())

rate_plot_df = (
    rate_wide
    .resample("M").last()
    .reset_index()
    .melt(id_vars="date", var_name="currency", value_name="rate")
    .dropna()
)

fig = px.line(
    rate_plot_df,
    x="date",
    y="rate",
    color="currency",
    title="Representative Short-Rate Proxies",
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_title=None,
    yaxis_title="Rate level",
    legend_title_text="Currency",
)
fig


,rate_ticker
USD,US0003M Index
EUR,EUR003M Index
GBP,BP0003M Index
JPY,JY0003M Index
CHF,SF0003M Index
AUD,ADBB3M Curncy
CAD,CDOR03 Index
NOK,NIBOR3M Index
SEK,STIB3M Index
HKD,HDDRC Curncy


In [15]:
if "USD" in rate_wide:
    carry_proxy = rate_wide.drop(columns=["USD"], errors="ignore").sub(rate_wide["USD"], axis=0)
else:
    carry_proxy = rate_wide.drop(columns=["USD"], errors="ignore") * np.nan

carry_plot_df = (
    carry_proxy
    .reindex(columns=[ccy for ccy in TARGET_CCYS if ccy in carry_proxy])
    .resample("M").last()
    .reset_index()
    .melt(id_vars="date", var_name="currency", value_name="rate_diff_vs_usd")
    .dropna()
)

fig = px.line(
    carry_plot_df,
    x="date",
    y="rate_diff_vs_usd",
    color="currency",
    title="Simple Carry Proxy: Local Short Rate Minus USD Short Rate",
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_title=None,
    yaxis_title="Rate differential vs USD",
    legend_title_text="Currency",
)
fig


In [16]:
monthly_fx_returns = np.expm1(spot_log_returns.resample("M").sum())
monthly_carry_proxy = carry_proxy.resample("M").last().shift(1)

carry_rows = []

for ccy in TARGET_CCYS:
    if ccy not in monthly_carry_proxy or ccy not in monthly_fx_returns:
        continue

    sample = pd.concat(
        [
            monthly_carry_proxy[ccy].rename("lagged_rate_diff"),
            monthly_fx_returns[ccy].rename("next_month_fx_return"),
        ],
        axis=1,
    ).dropna()

    if len(sample) < 24:
        continue

    high_carry = sample["lagged_rate_diff"] > sample["lagged_rate_diff"].median()

    carry_rows.append({
        "currency": ccy,
        "obs_months": len(sample),
        "average_rate_diff": sample["lagged_rate_diff"].mean(),
        "carry_fx_corr": sample["lagged_rate_diff"].corr(sample["next_month_fx_return"]),
        "avg_return_high_carry_months": sample.loc[high_carry, "next_month_fx_return"].mean(),
        "avg_return_low_carry_months": sample.loc[~high_carry, "next_month_fx_return"].mean(),
    })

carry_analysis = pd.DataFrame(carry_rows).set_index("currency").sort_values("average_rate_diff", ascending=False)

display_table(
    carry_analysis,
    {
        "obs_months": "{:,.0f}",
        "average_rate_diff": "{:.2f}",
        "carry_fx_corr": "{:.2f}",
        "avg_return_high_carry_months": "{:.2%}",
        "avg_return_low_carry_months": "{:.2%}",
    },
)


,obs_months,average_rate_diff,carry_fx_corr,avg_return_high_carry_months,avg_return_low_carry_months
currency,,,,,
BRL,233,10.60,-0.06,-0.68%,0.16%
MXN,233,4.61,0.02,-0.07%,-0.21%
CNH,233,2.17,-0.13,-0.12%,0.12%
CNY,233,1.21,-0.00,0.11%,0.04%
AUD,233,1.16,-0.01,0.08%,-0.05%
NOK,233,0.55,-0.09,-0.41%,0.14%
CAD,233,0.24,-0.16,-0.30%,0.20%
GBP,233,0.06,-0.14,-0.24%,-0.04%
HKD,233,-0.42,-0.02,0.01%,-0.01%


**How to explain the carry table**

This is not a production carry strategy. It is a quick sanity check: when a currency had a larger short-rate advantage versus USD, did the next month of spot performance tend to be better or worse? For high-carry EM, the answer can change by regime. High yield helps in calm markets, but it does not protect the currency when investors are de-risking or USD funding is scarce.


### 10. Literature-grounded currency factor diagnostics

The two papers suggest that preliminary FX analysis should not stop at individual currencies.

- Burnside, Eichenbaum, and Rebelo emphasize that carry and momentum are distinct currency phenomena. Momentum is based on recent currency performance, while carry is based on interest-rate or forward-discount information.
- Lustig, Roussanov, and Verdelhan argue that currency portfolios can be summarized by common factors: a broad dollar/common FX factor and a high-interest-rate-minus-low-interest-rate carry factor.

At this stage we are not running or recommending a trading strategy. The goal is diagnostic: do simple currency factors help explain the patterns we see in spot returns?


In [17]:
def spread_factor_from_signal(return_panel, signal_panel, top_n=3, bottom_n=3):
    rows = []
    common_dates = return_panel.index.intersection(signal_panel.index)

    for date in common_dates:
        returns = return_panel.loc[date].dropna()
        signals = signal_panel.loc[date].dropna()
        available = returns.index.intersection(signals.index)

        if len(available) < top_n + bottom_n:
            continue

        ranked = signals.loc[available].sort_values()
        low_ccys = ranked.head(bottom_n).index.tolist()
        high_ccys = ranked.tail(top_n).index.tolist()

        high_return = returns.loc[high_ccys].mean()
        low_return = returns.loc[low_ccys].mean()

        rows.append({
            "date": date,
            "factor_return": high_return - low_return,
            "high_leg_return": high_return,
            "low_leg_return": low_return,
            "high_ccys": ", ".join(high_ccys),
            "low_ccys": ", ".join(low_ccys),
        })

    if not rows:
        return pd.DataFrame(
            columns=["factor_return", "high_leg_return", "low_leg_return", "high_ccys", "low_ccys"]
        )

    return pd.DataFrame(rows).set_index("date").sort_index()


monthly_fx_panel = monthly_fx_returns.reindex(columns=TARGET_CCYS)

# Dollar/common FX factor proxy: equal-weighted average return across the currency universe.
dollar_factor_proxy = monthly_fx_panel.mean(axis=1, skipna=True).rename("Dollar factor proxy")

# Carry factor proxy: high-rate currencies minus low-rate currencies, using lagged rate differentials.
carry_factor_details = spread_factor_from_signal(
    return_panel=monthly_fx_panel,
    signal_panel=monthly_carry_proxy.reindex(columns=TARGET_CCYS),
    top_n=3,
    bottom_n=3,
)
carry_factor_proxy = carry_factor_details["factor_return"].rename("Carry factor proxy")

# Momentum factor proxy: recent winners minus recent losers, using prior-month spot returns.
momentum_signal = monthly_fx_panel.shift(1)
momentum_factor_details = spread_factor_from_signal(
    return_panel=monthly_fx_panel,
    signal_panel=momentum_signal,
    top_n=3,
    bottom_n=3,
)
momentum_factor_proxy = momentum_factor_details["factor_return"].rename("Momentum factor proxy")

currency_factor_returns = pd.concat(
    [dollar_factor_proxy, carry_factor_proxy, momentum_factor_proxy],
    axis=1,
).dropna(how="all")

factor_stats = []

for col in currency_factor_returns.columns:
    r = currency_factor_returns[col].dropna()
    if len(r) < 24:
        continue

    index = (1 + r).cumprod() * 100
    ann_return = index.iloc[-1] ** (12 / len(r)) / 100 ** (12 / len(r)) - 1

    factor_stats.append({
        "factor": col,
        "obs_months": len(r),
        "ann_return": ann_return,
        "ann_vol": r.std() * np.sqrt(12),
        "return_to_vol": ann_return / (r.std() * np.sqrt(12)) if r.std() != 0 else np.nan,
        "max_drawdown": max_drawdown(index),
    })

currency_factor_stats = pd.DataFrame(factor_stats).set_index("factor")

display_table(
    currency_factor_stats,
    {
        "obs_months": "{:,.0f}",
        "ann_return": "{:.2%}",
        "ann_vol": "{:.2%}",
        "return_to_vol": "{:.2f}",
        "max_drawdown": "{:.2%}",
    },
)


,obs_months,ann_return,ann_vol,return_to_vol,max_drawdown
factor,,,,,
Dollar factor proxy,234,-0.80%,6.95%,-0.12,-34.16%
Carry factor proxy,233,-1.20%,8.37%,-0.14,-41.27%
Momentum factor proxy,233,-3.10%,9.15%,-0.34,-58.06%


In [18]:
factor_index = (1 + currency_factor_returns).cumprod() * 100

factor_plot_df = (
    factor_index
    .reset_index()
    .melt(id_vars="date", var_name="factor", value_name="index")
    .dropna()
)

fig = px.line(
    factor_plot_df,
    x="date",
    y="index",
    color="factor",
    title="Preliminary Currency Factor Proxies, 100 = Start of Series",
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_title=None,
    yaxis_title="Cumulative index",
    legend_title_text="Factor proxy",
)
fig


In [19]:
factor_risk_corr = (
    currency_factor_returns
    .join(risk_changes.resample("M").sum(min_count=5), how="inner")
    .corr()
    .loc[currency_factor_returns.columns, key_factors]
)

fig = px.imshow(
    factor_risk_corr,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    labels={"x": "Risk or macro factor", "y": "Currency factor proxy", "color": "Corr"},
    title="Currency Factor Proxy Correlations with Risk and Macro Factors",
    text_auto=".2f",
)
fig.update_layout(template="plotly_white", height=420)
fig


In [20]:
leg_preview = pd.concat(
    [
        carry_factor_details.tail(8).assign(factor="Carry proxy"),
        momentum_factor_details.tail(8).assign(factor="Momentum proxy"),
    ]
)[["factor", "factor_return", "high_ccys", "low_ccys"]]

display_table(
    leg_preview,
    {"factor_return": "{:.2%}"},
)


,factor,factor_return,high_ccys,low_ccys
date,,,,
2025-11-30 00:00:00,Carry proxy,1.60%,"GBP, MXN, BRL","CHF, JPY, CNY"
2025-12-31 00:00:00,Carry proxy,-0.47%,"GBP, MXN, BRL","CHF, JPY, CNY"
2026-01-31 00:00:00,Carry proxy,1.18%,"GBP, MXN, BRL","CHF, JPY, CNY"
2026-02-28 00:00:00,Carry proxy,0.70%,"GBP, MXN, BRL","CHF, JPY, CNY"
2026-03-31 00:00:00,Carry proxy,-0.33%,"GBP, MXN, BRL","CHF, JPY, CNY"
2026-04-30 00:00:00,Carry proxy,1.80%,"GBP, MXN, BRL","CHF, JPY, CNY"
2026-05-31 00:00:00,Carry proxy,-0.31%,"GBP, MXN, BRL","CHF, JPY, CNY"
2026-06-30 00:00:00,Carry proxy,0.99%,"GBP, MXN, BRL","CHF, JPY, CNY"
2025-11-30 00:00:00,Momentum proxy,0.47%,"CNH, CNY, HKD","JPY, GBP, EUR"


The leg table is meant as an audit trail. It shows which currencies were in the high and low groups recently. This matters because factor behavior can change simply because the composition changes. For example, BRL and MXN may often appear in the high-rate group, while CHF and JPY often appear in the low-rate group.


### 10.1 Principal components of currency returns


In [21]:
pca_input = monthly_fx_panel.dropna(axis=1, thresh=60).dropna()
X = pca_input - pca_input.mean()

U, singular_values, Vt = np.linalg.svd(X.values, full_matrices=False)
explained_variance = singular_values**2 / np.sum(singular_values**2)

pca_explained = pd.DataFrame({
    "component": [f"PC{i}" for i in range(1, len(explained_variance) + 1)],
    "explained_variance": explained_variance,
    "cumulative_variance": np.cumsum(explained_variance),
}).head(8)

pc_scores = pd.DataFrame(
    U[:, :3] * singular_values[:3],
    index=pca_input.index,
    columns=["PC1", "PC2", "PC3"],
)

pca_loadings = pd.DataFrame(
    Vt[:3].T,
    index=pca_input.columns,
    columns=["PC1", "PC2", "PC3"],
)

# PCA signs are arbitrary. Flip PC1 so it lines up positively with the broad dollar/common FX proxy.
if pc_scores["PC1"].corr(dollar_factor_proxy.reindex(pc_scores.index)) < 0:
    pc_scores["PC1"] *= -1
    pca_loadings["PC1"] *= -1

display_table(
    pca_explained.set_index("component"),
    {
        "explained_variance": "{:.1%}",
        "cumulative_variance": "{:.1%}",
    },
)


,explained_variance,cumulative_variance
component,,
PC1,62.0%,62.0%
PC2,10.2%,72.2%
PC3,6.7%,78.9%
PC4,4.6%,83.5%
PC5,3.8%,87.3%
PC6,3.2%,90.5%
PC7,2.6%,93.2%
PC8,1.7%,94.9%


In [22]:
pca_plot_df = pca_loadings.copy()
pca_plot_df["currency"] = pca_plot_df.index
pca_plot_df = pca_plot_df.melt(
    id_vars="currency",
    var_name="component",
    value_name="loading",
)

fig = px.bar(
    pca_plot_df,
    x="currency",
    y="loading",
    color="loading",
    facet_col="component",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title="PCA Loadings of Monthly Currency Returns",
)
fig.update_layout(template="plotly_white", height=430)
fig.update_yaxes(matches=None)
fig


In [23]:
pca_factor_corr = (
    pc_scores
    .join(currency_factor_returns, how="inner")
    .corr()
    .loc[["PC1", "PC2", "PC3"], currency_factor_returns.columns]
)

display_table(
    pca_factor_corr,
    {col: "{:.2f}" for col in pca_factor_corr.columns},
    gradient=True,
)


,Dollar factor proxy,Carry factor proxy,Momentum factor proxy
PC1,0.99,0.36,-0.24
PC2,-0.08,0.81,-0.19
PC3,-0.04,0.14,-0.09


**How this connects to the papers**

The first principal component is usually a broad common FX move, similar in spirit to the dollar factor discussed by Lustig, Roussanov, and Verdelhan. A second or third component may separate high-beta/high-carry currencies from funding or managed currencies. The exact signs do not matter because PCA signs are arbitrary; the loading pattern is what matters.

This section also gives us a bridge to Burnside, Eichenbaum, and Rebelo: carry and momentum can both be visible in currencies, but they are not the same object. If the carry proxy, momentum proxy, and PCA components behave differently around VIX/DXY shocks, that is a useful early signal about what kind of FX variation the notebook is capturing.


### 11. Options-implied volatility and skew


In [24]:
options_raw = pd.concat([data["g10_options"], data["em_options"]], ignore_index=True)

OPTION_PAIR_BY_CCY = {
    "EUR": "EURUSD",
    "JPY": "USDJPY",
    "GBP": "GBPUSD",
    "CHF": "USDCHF",
    "CAD": "USDCAD",
    "AUD": "AUDUSD",
    "NZD": "NZDUSD",
    "SEK": "USDSEK",
    "NOK": "USDNOK",
    "HKD": "USDHKD",
    "MXN": "USDMXN",
    "BRL": "USDBRL",
    "CNY": "USDCNY",
    "CNH": "USDCNH",
}

VOL_1M_TICKERS = {ccy: f"{pair}V1M Curncy" for ccy, pair in OPTION_PAIR_BY_CCY.items()}
RR_1M_TICKERS = {ccy: f"{pair}25R1M Curncy" for ccy, pair in OPTION_PAIR_BY_CCY.items()}

def options_panel(ticker_map):
    available = {
        ccy: ticker
        for ccy, ticker in ticker_map.items()
        if ticker in set(options_raw["ticker"].unique())
    }

    panel = (
        options_raw[
            (options_raw["field"] == "PX_LAST")
            & (options_raw["ticker"].isin(available.values()))
        ]
        .pivot_table(index="date", columns="ticker", values="value", aggfunc="last")
        .rename(columns={ticker: ccy for ccy, ticker in available.items()})
        .sort_index()
    )

    return panel, available

vol_1m, available_vol_tickers = options_panel(VOL_1M_TICKERS)
rr_1m, available_rr_tickers = options_panel(RR_1M_TICKERS)

display(pd.Series(available_vol_tickers, name="1m_atm_vol_ticker").to_frame())


,1m_atm_vol_ticker
EUR,EURUSDV1M Curncy
JPY,USDJPYV1M Curncy
GBP,GBPUSDV1M Curncy
CHF,USDCHFV1M Curncy
CAD,USDCADV1M Curncy
AUD,AUDUSDV1M Curncy
NZD,NZDUSDV1M Curncy
SEK,USDSEKV1M Curncy
NOK,USDNOKV1M Curncy
HKD,USDHKDV1M Curncy


In [25]:
vol_plot_df = (
    vol_1m
    .reindex(columns=[ccy for ccy in TARGET_CCYS if ccy in vol_1m])
    .resample("M").last()
    .reset_index()
    .melt(id_vars="date", var_name="currency", value_name="implied_vol")
    .dropna()
)

fig = px.line(
    vol_plot_df,
    x="date",
    y="implied_vol",
    color="currency",
    title="1M FX Implied Volatility",
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_title=None,
    yaxis_title="1M implied volatility",
    legend_title_text="Currency",
)
fig


In [26]:
focus_rr = [ccy for ccy in ["MXN", "BRL", "CNH", "CNY", "JPY", "AUD", "CHF"] if ccy in rr_1m]

rr_plot_df = (
    rr_1m[focus_rr]
    .resample("M").last()
    .reset_index()
    .melt(id_vars="date", var_name="currency", value_name="risk_reversal")
    .dropna()
)

fig = px.line(
    rr_plot_df,
    x="date",
    y="risk_reversal",
    color="currency",
    title="1M 25-Delta Risk Reversals, Selected Currencies",
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_title=None,
    yaxis_title="25D risk reversal",
    legend_title_text="Currency",
)
fig


**Options read-through**

Implied volatility is the market price of uncertainty. Spikes usually coincide with drawdowns, policy shocks, or liquidity stress. Risk reversals add a directional layer: they show whether investors are paying more for upside or downside protection. The sign convention can vary by pair, so use the plot mainly for changes and stress episodes rather than treating every absolute level as directly comparable across currencies.


### 12. Focus currency review


In [27]:
focus_ccys = ["MXN", "BRL", "HKD", "CNY", "CNH"]

focus_summary = spot_stats.loc[[ccy for ccy in focus_ccys if ccy in spot_stats.index]].copy()

if not carry_analysis.empty:
    focus_summary = focus_summary.join(
        carry_analysis[["average_rate_diff", "carry_fx_corr"]],
        how="left",
    )

focus_summary = focus_summary.join(
    factor_corr[[col for col in ["DXY return", "VIX change", "Commodities return", "EM equities return"] if col in factor_corr.columns]],
    how="left",
)

display_table(
    focus_summary,
    {
        "ann_spot_return": "{:.2%}",
        "ann_vol": "{:.2%}",
        "return_to_vol": "{:.2f}",
        "max_drawdown": "{:.2%}",
        "best_day": "{:.2%}",
        "worst_day": "{:.2%}",
        "positive_days": "{:.1%}",
        "average_rate_diff": "{:.2f}",
        "carry_fx_corr": "{:.2f}",
        "DXY return": "{:.2f}",
        "VIX change": "{:.2f}",
        "Commodities return": "{:.2f}",
        "EM equities return": "{:.2f}",
    },
)


,name,ann_spot_return,ann_vol,return_to_vol,max_drawdown,best_day,worst_day,positive_days,obs,average_rate_diff,carry_fx_corr,DXY return,VIX change,Commodities return,EM equities return
currency,,,,,,,,,,,,,,,
MXN,Mexican peso,-2.37%,12.47%,-0.19,-61.36%,6.88%,-7.67%,51.8%,5082,4.61,0.02,-0.34,-0.48,0.30,0.34
BRL,Brazilian real,-4.22%,16.39%,-0.26,-73.61%,7.88%,-6.86%,49.5%,4742,10.60,-0.06,-0.31,-0.38,0.35,0.36
HKD,Hong Kong dollar,-0.03%,0.62%,-0.05,-1.28%,0.40%,-0.27%,45.9%,5082,-0.42,-0.02,-0.12,-0.05,0.06,0.15
CNY,"Chinese yuan, onshore",0.91%,3.11%,0.29,-14.79%,1.62%,-1.82%,49.4%,4708,1.21,-0.00,-0.28,-0.08,0.17,0.24
CNH,"Chinese yuan, offshore",-0.10%,4.16%,-0.02,-19.27%,2.02%,-2.71%,50.2%,4128,2.17,-0.13,-0.42,-0.18,0.22,0.27


### Research interpretation

**MXN:** The peso is usually one of the cleaner carry stories in EM FX. When volatility is contained, Mexico's rate premium can attract capital and support the currency. The vulnerability is that MXN is also liquid and widely used as an EM risk proxy, so it can sell off quickly when VIX, DXY, or US rates jump.

**BRL:** The real tends to behave like a high-beta mix of carry, commodities, domestic policy credibility, and global risk appetite. It can rally hard when carry is attractive and commodities are supportive, but it usually has larger drawdowns than G10 currencies because investors demand a higher risk premium.

**HKD:** The Hong Kong dollar should look structurally different because of the USD-linked exchange-rate system. Spot performance is intentionally constrained, so the interesting analysis is less about trend return and more about peg pressure, local liquidity, and rate differentials.

**CNY/CNH:** The yuan complex is managed rather than purely floating. That should reduce realized volatility relative to high-beta EM FX. CNH can move more freely offshore, especially in stress periods or when investors debate China's growth/policy outlook. Watch for step-like moves rather than continuous trend behavior.

**G10 block:** AUD, NZD, CAD, NOK, and SEK usually carry more global-growth and commodity beta. JPY and CHF can behave defensively, but their safe-haven role is not constant. EUR and GBP often sit between growth, rate differentials, and USD-cycle themes.


# Next steps

1. Add exact carry returns using forward points, not only rate proxies.
2. Split the sample into pre- and post-2020 periods to test whether inflation and rate volatility changed FX behavior.
3. Add event annotations for major stress periods such as the GFC, euro crisis, 2015 CNY devaluation, COVID, and the 2022 Fed shock.
4. Compare spot-only results with a simple carry basket: long high-rate currencies, short low-rate currencies, equal risk-weighted.
